In [1]:
code = '''import torch, torch.nn as nn
import snntorch as snn
from snntorch import surrogate

class SNN(nn.Module):
    def __init__(self, n_in=128, n_h=128, n_out=2, beta=0.9):
        super().__init__()
        g = surrogate.fast_sigmoid()
        self.fc1 = nn.Linear(n_in, n_h); self.lif1 = snn.Leaky(beta=beta, spike_grad=g)
        self.fc2 = nn.Linear(n_h, n_out); self.lif2 = snn.Leaky(beta=beta, spike_grad=g)
        self.drop = nn.Dropout(0.3)

    def forward(self, x):
        mem1 = self.lif1.init_leaky(); mem2 = self.lif2.init_leaky()
        out, hid = [], []
        for t in range(x.shape[0]):
            spk1, mem1 = self.lif1(self.fc1(x[t]), mem1)
            spk2, mem2 = self.lif2(self.fc2(self.drop(spk1)), mem2)
            out.append(spk2); hid.append(spk1)
        return torch.stack(out), torch.stack(hid)
'''
with open("snn_model.py", "w") as f:
    f.write(code)
print("saved")

saved


In [2]:
from snn_model import SNN
import snntorch.functional as SF
from snntorch import spikegen

In [3]:
import numpy as np, torch
from preprocess import band_power_features
from sklearn.model_selection import GroupShuffleSplit

z = np.load("data/processed/s01.npz")
F = band_power_features(z["X"]).astype(np.float32)
y = z["y"][:, 0]                      # 0 = valence, 1 = arousal
tid = z["trial_id"]

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(F, y, tid))

mu, sd = F[tr].mean(0), F[tr].std(0) + 1e-8        # statistics from TRAIN only
def scale(a):                                       # map to [0, 1] for rate coding
    return np.clip((a - mu) / sd, -3, 3) / 6 + 0.5

Xtr = torch.tensor(scale(F[tr]), dtype=torch.float32)
Xte = torch.tensor(scale(F[te]), dtype=torch.float32)
ytr = torch.tensor(y[tr]); yte = torch.tensor(y[te])
print(Xtr.shape, Xte.shape)

torch.Size([870, 128]) torch.Size([290, 128])


In [4]:
T = 25                                              # time steps per sample
model = SNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = SF.ce_rate_loss()

def evaluate(X, Y):
    model.eval()
    with torch.no_grad():
        spk, hid = model(spikegen.rate(X, num_steps=T))
    return SF.accuracy_rate(spk, Y), hid.mean().item()

for epoch in range(30):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(perm), 64):
        idx = perm[i:i+64]
        spk, _ = model(spikegen.rate(Xtr[idx], num_steps=T))
        loss = loss_fn(spk, ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 5 == 4:
        tr_acc, _ = evaluate(Xtr, ytr); te_acc, rate = evaluate(Xte, yte)
        print(f"epoch {epoch+1}  train {tr_acc:.3f}  test {te_acc:.3f}  hidden firing rate {rate:.3f}")

epoch 5  train 0.567  test 0.400  hidden firing rate 0.127
epoch 10  train 0.567  test 0.400  hidden firing rate 0.144
epoch 15  train 0.567  test 0.400  hidden firing rate 0.153
epoch 20  train 0.567  test 0.400  hidden firing rate 0.173
epoch 25  train 0.567  test 0.400  hidden firing rate 0.180
epoch 30  train 0.621  test 0.348  hidden firing rate 0.178
